# AlphaFold Ensemble Competition - Tournament Edition

This notebook screens a list of potential peptide binders using a 'Winner Stays On' (King of the Hill) tournament structure.
It evaluates pairs of binders against the target protein. The winner advances to face the next candidate on the list.

In [ ]:
# @title 1. Setup Environment
# @markdown Run this cell to install ColabFold and download necessary files.
import os
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Installing dependencies (this may take a few minutes)...")
    !pip install -q -U biopython py3Dmol "colabfold[alphafold] @ git+https://github.com/sokrypton/ColabFold"
    if not os.path.exists("af_competition.py"):
        print("Downloading af_competition.py...")
        !wget -q https://raw.githubusercontent.com/Drew-Thomson/AFcompetition/main/af_competition.py
    print("Setup complete! Please ensure you are using a GPU runtime (Runtime -> Change runtime type -> T4 GPU).")
else:
    print("Running locally. Dependencies assumed to be met.")

In [ ]:
# @title 2. (Optional) Mount Google Drive for persistent storage
# @markdown Run this to save your results permanently to your Google Drive.
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_OUTPUT_DIRECTORY = '/content/drive/MyDrive/AF_Competition_Results'
    os.makedirs(BASE_OUTPUT_DIRECTORY, exist_ok=True)
    print(f"Results will be saved to {BASE_OUTPUT_DIRECTORY}")
else:
    BASE_OUTPUT_DIRECTORY = './colabfold_results'

In [1]:
import os
import time
import py3Dmol
import numpy as np
from pathlib import Path
from af_competition import run_colabfold_async, analyze_binding, process_ensemble, optimize_threshold  # noqa: F401


## 1. Configuration
Define the target, binding site, and list of potential binders.

In [2]:

TARGET_SEQ = "SQIPASEQETLVRPKPLLLKLLKSVGAQKDTYTMKEVLFYLGQYIMTKRLYDAAQQHIVYCSNDLLGDLFGVPSFSVKEHRKIYTMIYRNLVVVNQQ"
BINDING_SITE_RESIDUES = [42, 84]

BINDER_CANDIDATES = [
    "ETFSDLWKLLPE",  # Ligand 1 (p53 wt)
    "ETFSDLAKLLPE",  # Ligand 1 (W->A mutant)
    "ETFSALWKLLPE",  # Ligand 1 (D->A mutant)
    "LTFEHYWAQLTS",  # Ligand 2 (PMI wt)
    "LTFEHYAAQLTS",  # Ligand 2 (W->A mutant)
    "LTFEHAWAQLTS",  # Ligand 2 (Y->A mutant)
]

NUM_SEEDS = 20
# BASE_OUTPUT_DIRECTORY is set in the Google Drive cell above.
# If running locally or without Drive, default to local folder:
if 'BASE_OUTPUT_DIRECTORY' not in locals():
    BASE_OUTPUT_DIRECTORY = "./colabfold_results"
MIN_PLDDT = 80.0
WARN_PLDDT_FAILURE_RATE = 0.25
NUM_RECYCLES = 20


## 2. Tournament Execution
Runs the 'Winner Stays On' tournament.

In [3]:
champion_idx = 0
champion_seq = BINDER_CANDIDATES[0]

last_match_dir = ""
last_match_stats = {}
winning_state_last_match = ""
majority_wins = (NUM_SEEDS // 2) + 1

print(f"Starting Tournament with {len(BINDER_CANDIDATES)} candidates. Early stop threshold: {majority_wins} wins.\n")

for challenger_idx in range(1, len(BINDER_CANDIDATES)):
    challenger_seq = BINDER_CANDIDATES[challenger_idx]
    run_name = f"tournament_match_{champion_idx}_vs_{challenger_idx}"
    
    print(f"--- Match {challenger_idx}: Champion [{champion_idx}] vs Challenger [{challenger_idx}] ---")
    
    # --- 1. Execute ColabFold Asynchronously ---
    process, ACTUAL_OUTPUT_DIR, log_file = run_colabfold_async(TARGET_SEQ, champion_seq, challenger_seq, BASE_OUTPUT_DIRECTORY, run_name=run_name, num_seeds=NUM_SEEDS)
    last_match_dir = ACTUAL_OUTPUT_DIR
    
    # --- 2. Monitor and Analyze in Real-Time ---
    import re
    analyzed_pdbs = set()
    seen_seeds = set()
    all_results_dict = {}
    early_stop_triggered = False
    
    while process.poll() is None:
        current_pdbs = set(Path(ACTUAL_OUTPUT_DIR).glob("*.pdb"))
        new_pdbs = current_pdbs - analyzed_pdbs
        
        newly_analyzed = 0
        for pdb in new_pdbs:
            try:
                res = analyze_binding(str(pdb), BINDING_SITE_RESIDUES, champion_seq, challenger_seq, 0.0)
                
                seed_match = re.search(r'seed_(\d+)', pdb.name)
                seed = seed_match.group(1) if seed_match else pdb.name
                
                if res["mean_plddt"] >= MIN_PLDDT:
                    all_results_dict[seed] = res
                else:
                    if seed not in seen_seeds:
                        print(f"  -> Model {seed} rejected (pLDDT: {res['mean_plddt']:.1f} < {MIN_PLDDT})")
                
                analyzed_pdbs.add(pdb)
                seen_seeds.add(seed)
                newly_analyzed += 1
            except Exception:
                # File might be partially written; skip and retry next loop
                pass
                
        if newly_analyzed > 0 and all_results_dict:
            all_results = list(all_results_dict.values())
            opt = optimize_threshold(all_results, min_thresh=2.0, max_thresh=8.0, step=0.25)
            stats = opt['stats']
            champ_wins = stats.get('lig1_wins', 0)
            challenger_wins = stats.get('lig2_wins', 0)
            
            print(f"Models parsed: {len(seen_seeds)}/{NUM_SEEDS} | Champion: {champ_wins} | Challenger: {challenger_wins}")
            
            if champ_wins >= majority_wins or challenger_wins >= majority_wins:
                print("  -> Insurmountable lead detected! Terminating AlphaFold early...")
                process.terminate()
                if 'log_file' in locals():
                    log_file.close()
                early_stop_triggered = True
                break
                
        time.sleep(5)  # Wait 5 seconds before checking again
        
    if not early_stop_triggered:
        # Parse any final straggler PDBs after process ends
        current_pdbs = set(Path(ACTUAL_OUTPUT_DIR).glob("*.pdb"))
        new_pdbs = current_pdbs - analyzed_pdbs
        for pdb in new_pdbs:
            try:
                res = analyze_binding(str(pdb), BINDING_SITE_RESIDUES, champion_seq, challenger_seq, 0.0)
                seed_match = re.search(r'seed_(\d+)', pdb.name)
                seed = seed_match.group(1) if seed_match else pdb.name
                if res["mean_plddt"] >= MIN_PLDDT:
                    all_results_dict[seed] = res
                else:
                    if seed not in seen_seeds:
                        print(f"  -> Model {seed} rejected (pLDDT: {res['mean_plddt']:.1f} < {MIN_PLDDT})")
                        seen_seeds.add(seed)
            except Exception:
                pass
        all_results = list(all_results_dict.values())
                
            
        if 'log_file' in locals() and not log_file.closed:
            log_file.close()
        
        if not all_results:
            print(f"Match void: No models passed pLDDT threshold {MIN_PLDDT}.")
            print(f"Champion [{champion_idx}] retains title by default.\n")
            winning_state_last_match = "Ligand 1"
            continue
            
        # Final optimization across all parsed results
        opt = optimize_threshold(all_results, min_thresh=2.0, max_thresh=8.0, step=0.25)
        stats = opt['stats']
    last_match_stats = stats
    
    champ_wins = stats.get('lig1_wins', 0)
    challenger_wins = stats.get('lig2_wins', 0)
    
    total_models = len(seen_seeds)
    valid_models = len(all_results_dict)
    invalid_models = total_models - valid_models

    if total_models > 0 and (invalid_models / total_models) >= WARN_PLDDT_FAILURE_RATE:
        failure_pct = (invalid_models / total_models) * 100
        print(f"\n⚠️ WARNING: High rate of low-confidence models detected.")
        print(f"  {invalid_models}/{total_models} ({failure_pct:.1f}%) models fell below the minimum pLDDT threshold of {MIN_PLDDT}.")
        print(f"  💡 TIP: Consider increasing 'NUM_RECYCLES' (currently {NUM_RECYCLES}) or lowering 'MIN_PLDDT'.")
    
    print(f"\nFinal Match Results (Valid Models: {len(all_results)})")
    print(f"Champion score: {champ_wins} | Challenger score: {challenger_wins}")
    
    if challenger_wins > champ_wins:
        print(f"Challenger [{challenger_idx}] defeats Champion [{champion_idx}]!")
        champion_idx = challenger_idx
        champion_seq = challenger_seq
        winning_state_last_match = "Ligand 2"
    elif champ_wins > challenger_wins:
        print(f"Champion [{champion_idx}] defends the title!")
        winning_state_last_match = "Ligand 1"
    else:
        # TIE BREAKER
        print("Scores tied! Proceeding to pLDDT tie-breaker...")
        if champ_wins == 0 and challenger_wins == 0:
            print("Neither ligand achieved exclusive binding in any valid models. Champion retains title by default.")
            winning_state_last_match = "Neither"
        else:
            champ_plddt = np.mean([r['mean_plddt'] for r in stats['lig1_results']])
            challenger_plddt = np.mean([r['mean_plddt'] for r in stats['lig2_results']])
            print(f"Champion Avg pLDDT: {champ_plddt:.1f} | Challenger Avg pLDDT: {challenger_plddt:.1f}")
            
            if challenger_plddt > champ_plddt:
                print(f"Challenger [{challenger_idx}] wins by tie-breaker!")
                champion_idx = challenger_idx
                champion_seq = challenger_seq
                winning_state_last_match = "Ligand 2"
            else:
                print(f"Champion [{champion_idx}] wins by tie-breaker!")
                winning_state_last_match = "Ligand 1"
    print("\n")
    
print("=== TOURNAMENT COMPLETE ===")
print(f"ULTIMATE CHAMPION: Candidate [{champion_idx}] ({champion_seq})")


Starting Tournament with 4 candidates. Early stop threshold: 11 wins.

--- Match 1: Champion [0] vs Challenger [1] ---
Models parsed: 1/20 | Champion: 0 | Challenger: 1
Models parsed: 2/20 | Champion: 1 | Challenger: 1
Models parsed: 3/20 | Champion: 1 | Challenger: 2
Models parsed: 4/20 | Champion: 1 | Challenger: 3
Models parsed: 5/20 | Champion: 1 | Challenger: 4
Models parsed: 6/20 | Champion: 2 | Challenger: 4
Models parsed: 7/20 | Champion: 2 | Challenger: 5
Models parsed: 8/20 | Champion: 2 | Challenger: 6
Models parsed: 9/20 | Champion: 3 | Challenger: 6
Models parsed: 10/20 | Champion: 3 | Challenger: 7
Models parsed: 11/20 | Champion: 3 | Challenger: 8
Models parsed: 12/20 | Champion: 4 | Challenger: 8
Models parsed: 13/20 | Champion: 5 | Challenger: 8
Models parsed: 14/20 | Champion: 6 | Challenger: 8
Models parsed: 15/20 | Champion: 7 | Challenger: 8
Models parsed: 16/20 | Champion: 8 | Challenger: 8
Models parsed: 17/20 | Champion: 9 | Challenger: 8
Models parsed: 18/20 | 

## 3. Visualization of the Final Match
Renders the highest confidence model of the Ultimate Champion winning its last match.

In [5]:
if not last_match_stats:
    print("No valid models were generated to visualize.")
elif winning_state_last_match == "Neither":
    print("Neither ligand bound in the final match, nothing to visualize.")
else:
    winning_results = []
    
    if winning_state_last_match == "Ligand 1":
        winning_results = last_match_stats.get('lig1_results', [])
    elif winning_state_last_match == "Ligand 2":
        winning_results = last_match_stats.get('lig2_results', [])
        
    if not winning_results:
        print("No structural models available for the winning state to render.")
    else:
        best_model = max(winning_results, key=lambda x: x["mean_plddt"])
        best_pdb = os.path.join(last_match_dir, best_model["pdb_file"])
        
        chain_lig1 = best_model.get("chain_lig1", "B")
        chain_lig2 = best_model.get("chain_lig2", "C")
        
        print(f"Visualizing Final Match. Winning State: {winning_state_last_match}")
        print(f"Best Model: {best_model['pdb_file']} (pLDDT: {best_model['mean_plddt']:.1f})")
        print(f"Target (Chain A) = Grey | Ligand 1 (Chain {chain_lig1}) = Blue | Ligand 2 (Chain {chain_lig2}) = Red")
        
        if os.path.exists(best_pdb):
            with open(best_pdb, 'r') as f:
                pdb_data = f.read()
                
            view = py3Dmol.view(width=800, height=600)
            view.addModel(pdb_data, 'pdb')
            
            # Target (Chain A)
            view.setStyle({'chain': 'A'}, {'cartoon': {'color': 'lightgray'}})
            # Ligand 1
            view.setStyle({'chain': chain_lig1}, {'cartoon': {'color': 'blue'}, 'stick': {'color': 'blue'}})
            # Ligand 2
            view.setStyle({'chain': chain_lig2}, {'cartoon': {'color': 'red'}, 'stick': {'color': 'red'}})
            
            view.zoomTo()
            view.show()
        else:
            print(f"Error: Cannot find {best_pdb} to visualize.")


Visualizing Final Match. Winning State: Ligand 2
Best Model: complex_unrelaxed_alphafold2_multimer_v3_model_1_seed_573343.pdb (pLDDT: 85.4)
Target (Chain A) = Grey | Ligand 1 (Chain B) = Blue | Ligand 2 (Chain C) = Red
Error: Cannot find ./colabfold_results/tournament_match_2_vs_3/complex_unrelaxed_alphafold2_multimer_v3_model_1_seed_573343.pdb to visualize.
